In [1]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [3]:
# llm = Ollama(model="dolphin-llama3:8b")

llm = Ollama(
    model="dolphin-llama3:8b",
    callbacks=[StreamingStdOutCallbackHandler()]
)

In [7]:
extract_prompt = PromptTemplate.from_template(
    "Extract the key points for searching from this query:\n\n{input}"
)
extract_chain = LLMChain(llm=llm, prompt=extract_prompt, output_key="extracted")

In [ ]:
action_prompt = PromptTemplate.from_template(
    "Based on this user request: \"{input}\", decide the correct action. Below the options and how you should answer is given\n"
    "Available actions: retrieve document info:rdi, reference content from document:rcfd, stop:stop\n\n"
    "Respond with only the action name."
)
action_chain = LLMChain(llm=llm, prompt=action_prompt, output_key="action")

In [10]:
initial_chain = SequentialChain(
    chains=[extract_chain, action_chain],
    input_variables=["input"],
    output_variables=["extracted", "action"],
    verbose=True
)

In [ ]:
from typing import Dict, List

def llm_flow(state: Dict, history: List[Dict] = None) -> Dict:
    if history is None:
        history = []

    # Append user or system-generated input to history
    input_source = state.get("input_source", "user")
    history.append({
        "role": "user" if input_source == "user" else "assistant",
        "content": state.get("input", "")
    })
    state["history"] = history

    # 🔁 Step 1: Action Selection
    if input_source == "user":
        action_prompt_text = f"""
            A user asked: "{input}"

            Available actions:
            - retrieve document info: rdi
            - reference content from document: rcfd
            - stop: stop

            🔥 IMPORTANT: Use `rdi` or `rcfd` ONLY if the user **explicitly mentions document**, **search**, or **wants information from files**.
            If the question can be answered directly or it's general knowledge (like 'Tell me about MongoDB'), return `stop`.

            Respond ONLY with one of: rdi, rcfd, or stop.
            """
    else:
        action_prompt_text = f"""
            Continue the reasoning based on internal step: "{state['input']}"

            Available actions:
            - retrieve document info: rdi
            - reference content from document: rcfd
            - stop: stop

            🔥 IMPORTANT: Use `rdi` or `rcfd` ONLY if the user **explicitly mentions document**, **search**, or **wants information from files**.
            If the question can be answered directly or it's general knowledge (like 'Tell me about MongoDB'), return `stop`.

            Respond ONLY with one of: rdi, rcfd, or stop.
            """

    action_prompt = PromptTemplate(
            input_variables=["input"],
            template=action_prompt_text
        )
    action_chain = LLMChain(llm=llm, prompt=action_prompt, output_key="action")
    action = action_chain.run(state)
    action = action.strip().lower()
    if action not in ("rdi", "rcfd", "stop"):
        action = "stop"  # Fallback if LLM didn't follow instructions
    state["action"] = action

    print(f"🔁 Action = {state['action']}")

    # 🔎 Step 2: Action Execution
    if state["action"] in ("rdi", "rcfd"):
        # Extract key points for search
        extract_prompt = PromptTemplate.from_template("""
            Extract key information from the input for semantic search embedding:
            {input}
            """)
        extract_chain = LLMChain(llm=llm, prompt=extract_prompt, output_key="extracted")
        extracted = extract_chain.run(state).strip()
        state["extracted"] = extracted

        # Embed and search
        embedded = embed_text(extracted)
        results = search_similar(embedded)
        state["retrieved_docs"] = results

        # 🧾 rdi = Show document list
        if state["action"] == "rdi":
            doc_titles = [doc["file_name"] for doc in results[:3]]
            doc_list = ", ".join(doc_titles) if doc_titles else "no matching documents"
            state["context"] = f"I found {len(results)} documents: {doc_list}."
            state["input"] = state["context"]
            state["input_source"] = "assistant"

        # 📘 rcfd = Summarize contents from top docs
        elif state["action"] == "rcfd":
            paired_docs = "\n\n".join(
                f"Document: {doc.get('file_name', 'Unknown')}\nContent:\n{doc.get('content', '')}"
                for doc in results
            )
            state["docs"] = paired_docs

            summary_prompt = PromptTemplate.from_template("""
                You are given content from several documents.
                Your task is to summarize the relevant insights for the user in a helpful and concise way.

                {docs}
                """)
            summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="context")
            summary = summary_chain.run(state).strip()

            state["context"] = summary
            state["input"] = summary
            state["input_source"] = "assistant"

    elif state["action"] == "stop":
        summary_prompt = PromptTemplate.from_template("""
                                                      {input}
                                                      """)
        summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="context")
        summary = summary_chain.run(state).strip()
        state["context"] = summary
        return (state, history)

    else:
        state["context"] = "Sorry, I didn't understand the action."
        state["input"] = "retry"
        state["input_source"] = "assistant"

    # 🔁 Recurse
    return llm_flow(state, history)


In [5]:
state = {
    "input": "Tell me about MongoDB",
    "input_source": "user"
}

history = []
final_state, history = llm_flow(state)
print("🔚 Final Output:\n", final_state["context"])


C:\Users\Niitro_musics\AppData\Local\Temp\ipykernel_14912\2976306277.py:50: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  action_chain = LLMChain(llm=llm, prompt=action_prompt, output_key="action")
C:\Users\Niitro_musics\AppData\Local\Temp\ipykernel_14912\2976306277.py:51: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  action = action_chain.run(state)


stop🔁 Action = stop
MongoDB is an open-source, cross-platform, document-oriented database program that allows for the storage of data in flexible, JSON-like documents. It was developed by MongoDB Inc., and it has since been adopted by many businesses due to its ability to handle unstructured or semi-structured data. The structure of a collection can vary from item to item, allowing for more flexibility than traditional relational databases.

The main features of MongoDB include:
1. Document-oriented storage: Stores data in flexible, JSON-like documents.
2. Indexing: Provides efficient indexing and querying capabilities.
3. Replication and sharding: Allows for horizontal scaling

KeyboardInterrupt: 